# Import Libraries

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go

from sklearn.cluster import KMeans

from mpl_toolkits.mplot3d import Axes3D

plt.style.use("dark_background")
sns.set_palette("viridis")

# Load Data

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/krupalpatel07/silver-market-intelligence-dataset/silver.csv")

df['Date'] = pd.to_datetime(df['Date'])

df = df.sort_values("Date")

df.set_index("Date", inplace=True)

# 1 Silver Market Structure

In [ ]:
plt.figure(figsize=(16,6))

plt.plot(df['Close'], linewidth=1.5)

plt.title("Silver Price Structure (2000–Present)")

plt.xlabel("Year")

plt.ylabel("Price USD")

plt.show()

# 2 Interactive Candlestick Chart

In [ ]:
fig = go.Figure(data=[go.Candlestick(
    x=df.index,
    open=df['Open'],
    high=df['High'],
    low=df['Low'],
    close=df['Close']
)])

fig.update_layout(title="Silver Market Structure")

fig.show()

# 3 Log Price Dynamics

In [ ]:
plt.figure(figsize=(15,6))

plt.plot(np.log(df['Close']))

plt.title("Log Price Behavior")

plt.show()

# 4 Daily Returns

In [ ]:
returns = df['Close'].pct_change()

plt.figure(figsize=(15,5))

plt.plot(returns)

plt.title("Silver Daily Returns")

plt.show()

# 5 Volatility Clustering

In [ ]:
volatility = returns.rolling(30).std()

plt.figure(figsize=(15,5))

plt.plot(volatility)

plt.title("Volatility Clustering")

plt.show()

# 6 Distribution of Returns

In [ ]:
sns.histplot(returns.dropna(), bins=120, kde=True)

plt.title("Return Distribution")

plt.show()

# 7 Moving Average Trend

In [ ]:
df['MA50'] = df['Close'].rolling(50).mean()
df['MA200'] = df['Close'].rolling(200).mean()

plt.figure(figsize=(16,6))

plt.plot(df['Close'])
plt.plot(df['MA50'])
plt.plot(df['MA200'])

plt.title("Trend Structure")

plt.show()

# 8 Market Regime Detection

In [ ]:
features = pd.DataFrame()

features['returns'] = returns
features['volatility'] = returns.rolling(20).std()

features = features.dropna()

kmeans = KMeans(n_clusters=3)

features['regime'] = kmeans.fit_predict(features)

## Regime Visualization

In [ ]:
plt.figure(figsize=(15,6))

plt.scatter(features.index,
            df.loc[features.index]['Close'],
            c=features['regime'],
            cmap='plasma')

plt.title("Market Regime Detection")

plt.show()

# 9 Seasonality Heatmap

In [ ]:
df['Year'] = df.index.year
df['Month'] = df.index.month

pivot = df.pivot_table(values='Close', index='Year', columns='Month')

sns.heatmap(pivot, cmap="viridis")

plt.title("Seasonality Map")

plt.show()

# 10 Drawdown Analysis

In [ ]:
returns = df['Close'].pct_change()

cum = (1 + returns).cumprod()

peak = cum.cummax()

drawdown = (cum - peak) / peak

plt.figure(figsize=(15,6))

plt.plot(drawdown)

plt.title("Drawdown Behavior")

plt.show()

# 11 3D Market Surface

In [ ]:
fig = plt.figure(figsize=(10,7))

ax = fig.add_subplot(111, projection='3d')

x = np.arange(len(df))
y = df['Close']
z = df['Volume'] if 'Volume' in df else df['Close'].rolling(5).std()

ax.scatter(x, y, z)

ax.set_xlabel("Time")
ax.set_ylabel("Price")
ax.set_zlabel("Market Activity")

plt.title("3D Silver Market Surface")

plt.show()

# 12 Interactive Silver Dashboard

In [ ]:
fig = px.line(df, x=df.index, y='Close',
              title="Interactive Silver Price Explorer")

fig.show()